In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.linalg import solve_banded
import pandas as pd
import time

# ── Parámetros del problema (idénticos a MELIN) ──────────────
alpha  = 0.2        # Constante de difusión térmica [m²/s^(1/2)]
L      = 1.0        # Ancho de la pared [m]
N      = 10         # Número de divisiones espaciales
dx     = L / N      # Paso espacial
T0     = 100.0      # Temperatura inicial [°C]

# ── Parámetros temporales ────────────────────────────────────
dt     = 0.05       # Paso de tiempo [s]  (mismo que MELIN h=0.05)
t_end  = 30.0       # Tiempo final [s]
NPT    = int(t_end / dt)   # Número de pasos de tiempo

# ── Número de Fourier (parámetro de estabilidad) ─────────────
r = alpha**2 * dt / dx**2

# ── Grid espacial ────────────────────────────────────────────
x = np.linspace(0, L, N+1)   # N+1 puntos: x_0, x_1, ..., x_N

print(f"α     = {alpha}")
print(f"l     = {L} m")
print(f"N     = {N}  nodos")
print(f"Δx    = {dx}")
print(f"Δt    = {dt}")
print(f"NPT   = {NPT} pasos de tiempo")
print(f"r     = α²Δt/Δx² = {r:.4f}")
print()
print("Crank-Nicolson es incondicionalmente estable → cualquier r es válido ✔")
print(f"(r = {r:.4f}, equivalente al MELIN: r = {alpha**2 * 0.05 / dx**2:.4f})")

α     = 0.2
l     = 1.0 m
N     = 10  nodos
Δx    = 0.1
Δt    = 0.05
NPT   = 600 pasos de tiempo
r     = α²Δt/Δx² = 0.2000

Crank-Nicolson es incondicionalmente estable → cualquier r es válido ✔
(r = 0.2000, equivalente al MELIN: r = 0.2000)


In [2]:
def construir_matrices(N, r):
    """
    Construye las matrices A (implícita) y B (explícita) del sistema
    tridiagonal de Crank-Nicolson para la ecuación de difusión 1D.

    Frontera x=0  : Dirichlet  T=0  → fila 0 de identidad
    Nodos int.    : i=1,...,N-1 → esquema CN estándar
    Frontera x=l  : Neumann ∂T/∂x=0 usando T_{N+1}=(4T_N-T_{N-1})/3
    """
    size = N + 1   # índices 0..N

    A = np.zeros((size, size))
    B = np.zeros((size, size))

    # ── Fila 0: Dirichlet T(0,t)=0 ──────────────────────────
    A[0, 0] = 1.0
    B[0, 0] = 0.0          # RHS siempre 0 → T_0^{n+1} = 0

    # ── Filas interiores i = 1 … N-1 ────────────────────────
    for i in range(1, N):
        A[i, i-1] = -r/2
        A[i, i  ] =  1 + r
        A[i, i+1] = -r/2

        B[i, i-1] =  r/2
        B[i, i  ] =  1 - r
        B[i, i+1] =  r/2

    # ── Fila N: Neumann (∂T/∂x=0 en x=l) ───────────────────
    # Sustituyendo T_{N+1} = (4T_N - T_{N-1})/3 en el esquema CN:
    # Lado implícito:
    #   -r/2 * T_{N-1}^{n+1}  +  (1+r)*T_N^{n+1}  -  r/2 * (4T_N-T_{N-1})/3
    # = -r/2*(1-1/3)*T_{N-1}^{n+1}  + (1+r - 2r/3)*T_N^{n+1}
    # = -r/3 * T_{N-1}^{n+1}  +  (1 + r/3)*T_N^{n+1}
    A[N, N-1] = -r/3
    A[N, N  ] =  1 + r/3

    # Lado explícito (análogo):
    B[N, N-1] =  r/3
    B[N, N  ] =  1 - r/3

    return A, B


A, B = construir_matrices(N, r)

print("Matriz A (implícita) — primeras 5×5:")
print(np.round(A[:5, :5], 4))
print()
print("Matriz B (explícita) — primeras 5×5:")
print(np.round(B[:5, :5], 4))

Matriz A (implícita) — primeras 5×5:
[[ 1.   0.   0.   0.   0. ]
 [-0.1  1.2 -0.1  0.   0. ]
 [ 0.  -0.1  1.2 -0.1  0. ]
 [ 0.   0.  -0.1  1.2 -0.1]
 [ 0.   0.   0.  -0.1  1.2]]

Matriz B (explícita) — primeras 5×5:
[[0.  0.  0.  0.  0. ]
 [0.1 0.8 0.1 0.  0. ]
 [0.  0.1 0.8 0.1 0. ]
 [0.  0.  0.1 0.8 0.1]
 [0.  0.  0.  0.1 0.8]]


In [3]:
# ── Condición inicial ────────────────────────────────────────
T_cn        = np.full(N+1, T0)   # T_i(t=0) = 100 para i=1..N
T_cn[0]     = 0.0                 # Dirichlet T(0,0)=0

# ── Almacenamiento de la solución ───────────────────────────
t_vals      = np.arange(0, t_end + dt, dt)   # tiempos de salida
T_hist      = np.zeros((N+1, len(t_vals)))    # T[nodo, tiempo]
T_hist[:, 0] = T_cn.copy()

# ── Factorización LU de A (solo una vez, fuera del bucle) ───
# np.linalg.solve la hace internamente; para mayor eficiencia
# en sistemas grandes conviene usar scipy.linalg.lu_factor/solve
from scipy.linalg import lu_factor, lu_solve
lu, piv = lu_factor(A)

# ── Cronómetro ───────────────────────────────────────────────
t_inicio_cn = time.perf_counter()

# ── Bucle temporal ───────────────────────────────────────────
T_actual = T_cn.copy()
for k in range(1, len(t_vals)):
    rhs           = B @ T_actual       # lado derecho
    rhs[0]        = 0.0               # Dirichlet: T_0 siempre 0
    T_nuevo       = lu_solve((lu, piv), rhs)
    T_nuevo[0]    = 0.0               # reforzar Dirichlet
    T_hist[:, k]  = T_nuevo
    T_actual      = T_nuevo

t_fin_cn   = time.perf_counter()
t_ejec_cn  = t_fin_cn - t_inicio_cn

# ── Temperatura en la frontera aislada (x=l) ────────────────
T_frontera_cn = (4*T_hist[N, :] - T_hist[N-1, :]) / 3.0

print(f"Integración completada: {len(t_vals)} pasos")
print(f"=========================================")
print(f"  Tiempo de ejecución : {t_ejec_cn:.6f} segundos")
print(f"  Archivo generado    : CN.dat")
print(f"=========================================")

Integración completada: 601 pasos
  Tiempo de ejecución : 0.007867 segundos
  Archivo generado    : CN.dat


In [4]:
# ── CN.dat: columnas = [t, T_1..T_N, T_{N+1}] ───────────────
datos_cn = np.column_stack([
    t_vals,
    T_hist[:N, :].T,       # nodos 1..N (sin el nodo 0 que es siempre 0)
    T_frontera_cn          # temperatura virtual en x=l
])

# Encabezado con nombres de columnas
header = "t" + "".join([f"      T{i+1}" for i in range(N)]) + "  T_frontera"
np.savetxt('CNPY.dat', datos_cn, fmt='%10.4f', header=header, comments='')

print("Primeras 5 filas de CNPY.dat:")
df_preview = pd.DataFrame(
    datos_cn[:5],
    columns=["t"] + [f"T{i+1}" for i in range(N)] + ["T_frontera"]
)
display(df_preview.round(4))

Primeras 5 filas de CNPY.dat:


,t,T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,T_frontera
0,0.00,0.0,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0,100.0
1,0.05,0.0,83.2160,98.5915,99.8818,99.9901,99.9992,99.9999,100.0000,100.0000,100.0,100.0
2,0.10,0.0,71.6298,95.2383,99.4006,99.9329,99.9930,99.9993,99.9999,100.0000,100.0,100.0
3,0.15,0.0,63.2916,91.2230,98.4477,99.7729,99.9704,99.9964,99.9996,100.0000,100.0,100.0
4,0.20,0.0,57.0580,87.1400,97.0988,99.4680,99.9157,99.9880,99.9984,99.9998,100.0,100.0
